# PaMCP (Tool-as-a-Service) — reproducing the experimental evaluation

This notebook re-runs the evaluation reported in Section 8 of *Tool-as-a-Service: Provenance-Aware Tool Discovery for
LLM Agents in Model Context Protocol Ecosystems*, following the experimental protocol (series E1–E22, research
questions RQ1–RQ4), and compares every result with the numbers reported in the paper.

**What you need:** a standard Colab CPU runtime. No GPU, no API key and no MCP server are needed. Every LLM output
used by the evaluation (the analyst's goal contracts, the two tool labellings, MCP-Zero's requests and server summaries,
COLT's training queries, ToolScope's equivalence judgements, the calibration answers) was recorded once and is
replayed from `data/llm_records/`; the prompts are in `docs/llm_protocols/`.

**How to use it:** run the cells from top to bottom (*Runtime → Run all*). Step 3 lets you choose which series to
re-run; with the defaults the whole notebook takes about **75 minutes** on a free Colab CPU
runtime. Results go to `results/reproduced/`; the paper's results in `results/final/` are never overwritten.

| Step | What it does | Time |
|---|---|---|
| 0 | Clone the repository at the evaluated branch, check the Python packages | ~1 min |
| 1 | Check that data, labels and recorded LLM outputs match the hashes recorded with the paper's results | <1 min |
| 2 | Show what the recorded LLM outputs look like | <1 min |
| 3 | Re-run the selected series (one entry point: `experiments/final/run_all.py`) | see table in Step 3 |
| 4 | Compare every reproduced result file with the paper's (`experiments/final/compare_results.py`) | <1 min |
| 5 | Headline numbers side by side, confirmatory tests, figures | <1 min |

**Evidence classes.** Each result file states its class: MEASURED (benchmark data), LABELLED (scored against the
independent labels $J_B$), SIMULATED (verifier or attacker simulated from labels $J_A$, scored against $J_B$),
SYNTHETIC (generated registries). Series that need executed MCP servers or human annotators (E13, observed effects in
E3, executed injected actions in E14, the human typing study of E2, E19, E21) were not run in the paper and are not
run here.

## Step 0 — Get the code and data
`COMMIT` pins the exact code version; leave it empty to use the head of the evaluation branch.

In [ ]:
#@title Settings
REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"  #@param {type:"string"}
BRANCH = "research/pamcp-final-evaluation"  #@param {type:"string"}
COMMIT = ""  #@param {type:"string"}
WORKDIR = "/content/taas-provenance-discovery"  #@param {type:"string"}
OUT = "results/reproduced"  #@param {type:"string"}

In [ ]:
import os, subprocess, sys, json, time, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB and os.environ.get("PAMCP_REPO"):          # local use: point PAMCP_REPO at an existing checkout
    WORKDIR = os.environ["PAMCP_REPO"]
root = Path(WORKDIR)
if not (root / ".git").exists():
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--single-branch", REPO_URL, str(root)], check=True)
if COMMIT:
    subprocess.run(["git", "-C", str(root), "checkout", "-q", COMMIT], check=True)
os.chdir(root)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository:", root, "\ncommit:    ", head)
print(subprocess.check_output(["git", "log", "-1", "--format=%cd  %s", "--date=iso"], text=True))

In [ ]:
# The evaluation needs numpy, scipy, scikit-learn and matplotlib (preinstalled on Colab).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import numpy, scipy, sklearn, matplotlib
here = {"python": sys.version.split()[0], "numpy": numpy.__version__, "scipy": scipy.__version__,
        "sklearn": sklearn.__version__, "matplotlib": matplotlib.__version__}
reference = {"python": "3.13.18", "numpy": "2.5.3", "scipy": "1.18.1", "sklearn": "1.9.1", "matplotlib": "3.11.2"}
print("this runtime:          ", here)
print("paper's results, with: ", reference, "(2 vCPUs)")
print("CPUs here:", os.cpu_count())

## Step 1 — Integrity of the inputs
Each result file of the paper records the SHA-256 of the merged registry, of the two labellings and of every recorded
LLM output it could read. This cell recomputes them on the cloned files.

In [ ]:
import hashlib, glob
def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()

rows, bad = [], []
for f in sorted(glob.glob("results/final/*.json")):
    m = json.load(open(f))["manifest"]
    files = {**m.get("inputs", {}), **m.get("labels", {}), **m.get("llm_records", {})}
    ok = sum(1 for p, h in files.items() if Path(p).exists() and sha(p) == h)
    miss = [p for p, h in files.items() if not Path(p).exists() or sha(p) != h]
    bad += [(Path(f).stem, p) for p in miss]
    rows.append((Path(f).stem, m["commit"][:7], len(files), ok))
print(f"{'result file':22} {'commit':8} {'inputs':>6} {'match':>6}")
for r in rows:
    print(f"{r[0]:22} {r[1]:8} {r[2]:6d} {r[3]:6d}")
print("\nall recorded inputs match" if not bad else f"\n{len(bad)} inputs differ from the recorded hashes:", bad[:10])

## Step 2 — The recorded LLM outputs
Three examples: the analyst's goal contract for a LiveMCPBench task, the two independent labellings of one tool
($J_A$ drives the engine's typing and the simulated verifier; $J_B$, produced blind, is used only for scoring), and an
MCP-Zero request.

In [ ]:
def first_jsonl(pattern):
    p = sorted(glob.glob(pattern))[0]
    with open(p) as fh:
        return p, json.loads(fh.readline())

p, rec = first_jsonl("data/llm_records/analyst_sonnet_A*.jsonl")
print("analyst goal contract —", p); print(json.dumps(rec, indent=1)[:1500], "\n")
silver = json.load(open("data/llm_records/silver_labels.json"))
tool = "opgg-mcp::lol-summoner-renewal"
a = silver.get(tool) if isinstance(silver, dict) else None
print("labels A (", tool, "):", json.dumps(a, indent=1)[:600] if a else "see silver_labels.json")
for part in sorted(glob.glob("data/llm_records/reference_labels_opus_part*.jsonl")):
    for line in open(part):
        r = json.loads(line)
        if tool in json.dumps(r):
            print("labels B:", json.dumps(r, indent=1)[:600]); break
p, rec = first_jsonl("data/llm_records/mcpzero_requests_sonnet_part*.jsonl")
print("\nMCP-Zero request —", p); print(json.dumps(rec, indent=1)[:800])

## Step 3 — Re-run the experiments
Tick the series to re-run. Times were measured on two virtual CPUs from a fresh clone; a Colab CPU runtime also has
two, but its CPUs may be slower.
E22 and the report use the results of E4, E8, E9, E10, E14 and E18: series you do not re-run are copied from the
paper's results, and this is logged in `run_log.json` and excluded from the comparison in Step 4.

| Series | Content (paper) | Minutes |
|---|---|---|
| E1-E3 | Registry characterisation, typing agreement, declared and lexical effects (Sections 8.2, 8.3) | <1 |
| E4-E7 | Exposure under policy encodings, forged metadata, imperfect verifier, trust policy (Fig. 9) | 13 |
| E8 | Single-tool retrieval and diagnosis (Fig. 10) | 15 |
| E9-E10 | Set retrieval and set validity (Table 8) | 13 |
| E11-E12 | Capability gap, model-profile calibration (Fig. 8) | <1 |
| E14-E18 | Prompt-injection reachability, 2^3 mechanism ablation (Table 9, Fig. 11) | 20 |
| E16 | Registry and model evolution (Section 8.6) | 1 |
| E15 | Cost and scalability, optimality gap (Section 8.6) | 7 (to 10,000 tools); 55 (to 100,000) |
| E17-E20 | Refinement, cost-weight sensitivity (Section 8.6) | 6 |
| E22 | Confirmatory family F1–F7 with Holm correction (Table 10) | <1 |
| report | Markdown tables, figures, the paper's LaTeX tables | <1 |

E15 builds registries of up to `E15_MAX_N` cloned tools. The paper goes to 100,000, where building the engine takes
about 35 minutes and 3 GB of memory (the pairwise risk relation is built eagerly); 10,000 keeps the step short.

In [ ]:
#@title Series to re-run
E1_E3 = True  #@param {type:"boolean"}
E4_E7 = True  #@param {type:"boolean"}
E8 = True  #@param {type:"boolean"}
E9_E10 = True  #@param {type:"boolean"}
E11_E12 = True  #@param {type:"boolean"}
E14_E18 = True  #@param {type:"boolean"}
E16 = True  #@param {type:"boolean"}
E15 = True  #@param {type:"boolean"}
E15_MAX_N = 10000  #@param [1000, 10000, 100000] {type:"raw"}
E17_E20 = True  #@param {type:"boolean"}
E22 = True  #@param {type:"boolean"}
REPORT = True  #@param {type:"boolean"}

In [ ]:
chosen = [name for name, flag in [("E1-E3", E1_E3), ("E4-E7", E4_E7), ("E8", E8), ("E9-E10", E9_E10),
                                  ("E11-E12", E11_E12), ("E14-E18", E14_E18), ("E16", E16), ("E15", E15),
                                  ("E17-E20", E17_E20), ("E22", E22), ("report", REPORT)] if flag]
print("re-running:", ", ".join(chosen))
cmd = [sys.executable, "experiments/final/run_all.py", "--out", OUT, "--only", ",".join(chosen),
       "--e15-max-n", str(E15_MAX_N)]
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nfinished in {(time.time() - t0) / 60:.1f} min, exit code {proc.returncode}")

## Step 4 — Compare with the paper's results
Every number, flag and string of each re-run result file is compared with the same entry in `results/final/`
(the manifest and wall-clock measurements, which depend on the machine, excluded). *Identical up to rounding* means
relative differences below $10^{-6}$. The evaluation is deterministic (fixed seeds, no dependence on Python's hash
seed), so with the library versions of Step 0 every re-run file should be *identical*; other library versions can
introduce small numerical differences. A reduced E15 run (`E15_MAX_N` below 100,000) reports the larger registry
sizes as missing.

In [ ]:
from IPython.display import Markdown, display
subprocess.run([sys.executable, "experiments/final/compare_results.py", OUT], check=True, capture_output=True)
display(Markdown(open(Path(OUT) / "comparison.md").read()))

## Step 5 — Headline numbers, confirmatory tests and figures
The numbers the paper's text relies on, read from the paper's result files and from this run (series that were not
re-run are marked as such).

In [ ]:
import pandas as pd

def get(res_dir, name, *path):
    p = Path(res_dir) / f"{name}.json"
    if not p.exists():
        return None
    x = json.load(open(p))
    for k in path:
        x = x[k]
    return x

H = [  # (paper section, quantity, result file, path, scale)
    ("8.2 E1", "tools not read-only (labels B)", "E1_registry", ("results", "merged", "non_read_only_B"), 1),
    ("8.3 E4", "Exp@5, no filter (%)", "E4_exposure", ("results", "1.0", "pamcp_rrf|no_filter", "Exp@5"), 100),
    ("8.3 E4", "Exp@5, verified evidence (%)", "E4_exposure", ("results", "1.0", "pamcp_rrf|verified_evidence", "Exp@5"), 100),
    ("8.3 E4", "Util@5, verified evidence (%)", "E4_exposure", ("results", "1.0", "pamcp_rrf|verified_evidence", "Util@5"), 100),
    ("8.3 E5", "Exp@5 closed-world annotations, delta 0.1 (%)", "E5_forgery", ("results", "both|0.1", "pamcp_rrf|closed_world_annotations", "Exp@5"), 100),
    ("8.3 E6", "Exp@5 verified, FAR 5% random, delta 0.2 (%)", "E6_verifier", ("results", "0.2|random|0.0|1.0|0.05", "pamcp_rrf|verified_evidence", "Exp@5"), 100),
    ("8.4 E8", "Hit@1 goal contract, 81 tools (%)", "E8_single_tool", ("results", "sample|B1", "pamcp_contract", "hit1", "mean"), 100),
    ("8.4 E8", "Hit@1 goal contract, 569 tools (%)", "E8_single_tool", ("results", "sample|B1m", "pamcp_contract", "hit1", "mean"), 100),
    ("8.4 E8", "Hit@1 MCP-Zero, 569 tools (%)", "E8_single_tool", ("results", "sample|B1m", "mcp_zero", "hit1", "mean"), 100),
    ("8.4 E9", "group recall PaMCP K=5", "E9_sets", ("summary", "pamcp@5", "recall", "mean"), 1),
    ("8.4 E9", "group recall per-subtask", "E9_sets", ("summary", "per_subtask[analyst]", "recall", "mean"), 1),
    ("8.4 E10", "joint feasibility PaMCP (%)", "E10_validity", ("summary", "pamcp(K=10)", "joint_feasible"), 100),
    ("8.2 E11", "|K| review scenario, n=3,000", "E11_gap", ("results", "sonnet|0.9|3000|tolerant|gap", "size"), 1),
    ("8.2 E12", "Haiku calibration accuracy (%)", "E12_profile", ("results", "haiku", "overall_accuracy"), 100),
    ("8.5 E14", "reachable attacks, Full PaMCP (%)", "E14_injection", ("attack_rates", "least_privilege|Full_PaMCP", "rate"), 100),
    ("8.5 E14", "reachable attacks, RAG-MCP top-5 (%)", "E14_injection", ("attack_rates", "least_privilege|rag_mcp@5|unfiltered", "rate"), 100),
    ("8.5 E18", "Shapley of S on recall", "E18_factorial", ("factorial", "least_privilege|recall", "shapley", "S"), 1),
    ("8.6 E16", "cached toolkits invalidated per event (%)", "E16_evolution", ("results", "invalidated_exact_share"), 100),
    ("8.6 E17", "certified at once, K=5 (%)", "E17_refinement", ("results", "K=5|k=0", "valid_rate"), 100),
    ("8.6 E20", "toolkits changed by gamma=0 (%)", "E20_sensitivity", ("results", "gamma=0.0", "changed_share"), 100),
    ("8.6 E15", "greedy/exact cost, 5 goals", "E15_optimality_gap", ("results", "5", "greedy_over_exact_mean"), 1),
    ("8.6 E15", "abstentions at 10,000 cloned tools", "E15_scalability", ("results", "10000", "abstentions"), 1),
]
copied = set()
if (Path(OUT) / "run_log.json").exists():
    for r in json.load(open(Path(OUT) / "run_log.json")):
        copied |= set(r.get("copied_from_reference", []))
rows = []
for sec, q, f, path, s in H:
    a = get("results/final", f, *path)
    b = None if f in copied else get(OUT, f, *path)
    match = "not re-run" if b is None else ("yes" if a is not None and abs(a - b) <= 1e-9 + 1e-6 * abs(a) else "NO")
    rows.append({"section": sec, "quantity": q, "paper": None if a is None else round(a * s, 3),
                 "this run": None if b is None else round(b * s, 3), "match": match})
pd.set_option("display.max_colwidth", 60)
display(pd.DataFrame(rows))

In [ ]:
ref = get("results/final", "E22_statistics", "family")
fam = None if "E22_statistics" in copied else get(OUT, "E22_statistics", "family")
if fam is None:
    print("E22 was not re-run: the paper's values are shown")
    fam = ref
display(pd.DataFrame([{"ID": k, "comparison": v["test"], "n": v["n"], "diff": round(v["diff"], 3),
                       "95% CI": [round(c, 3) for c in v["ci"]], "p": round(v["p"], 4), "p (Holm)": round(v["p_holm"], 4),
                       "paper p (Holm)": round(ref[k]["p_holm"], 4)} for k, v in fam.items()]))

In [ ]:
from IPython.display import Image
figs = Path(OUT) / "figures"
for stem in ["paper_eligibility", "paper_retrieval", "paper_profile_gap", "paper_factorial", "paper_scalability"]:
    p = figs / f"{stem}.png"
    if p.exists():
        print(stem); display(Image(filename=str(p), width=900))

## Where things are
| Series | Script (`experiments/final/`) | Paper |
|---|---|---|
| E1–E3 | `e01_e03_registry_typing_effects.py` | 8.2, 8.3 |
| E4–E7 | `e04_e07_exposure.py`, `e04_admitted.py` | 8.3, Fig. 9 |
| E8 | `e08_single_tool.py`, `e08_diagnostic.py` | 8.4, Fig. 10 |
| E9–E10 | `e09_e10_sets.py`, `set_evaluator.py` (independent evaluator) | 8.4, Table 8 |
| E11–E12 | `e11_e12_profile.py` | 8.2, Fig. 8 |
| E14, E18 | `e14_e18_factorial.py` | 8.5, Table 9, Fig. 11 |
| E15, E16 | `e15_e16_scale_evolution.py` | 8.6 |
| E17, E20 | `e17_e20_refine_sensitivity.py` | 8.6 |
| E22 | `e22_statistics.py` | Table 10 |
| shared | `fe.py` (data, labels, metrics, statistics, manifests), `methods.py` (PaMCP cells and baselines) | 8.1 |

The engine (capability model, Algorithms 1–3, certificate) is in `src/taas/`. Methods, deviations from the protocol
and baseline adaptations: `docs/FINAL_EVALUATION_METHODS.md`; the preregistered confirmatory family:
`docs/FINAL_EVALUATION_PLAN.md`; all findings: `docs/FINAL_EVALUATION.md`.

In [ ]:
#@title Optional: unit tests of the engine (about a minute)
RUN_TESTS = False  #@param {type:"boolean"}
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "hypothesis"], check=True)
    subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"])

In [ ]:
#@title Optional: download this run (results, logs, comparison) as a zip
DOWNLOAD = False  #@param {type:"boolean"}
if DOWNLOAD:
    z = shutil.make_archive("/content/pamcp_reproduced", "zip", OUT)
    if IN_COLAB:
        from google.colab import files
        files.download(z)
    print(z)